In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from dataclasses import dataclass
from lunanav.constants import GM_MOON, R_MOON, RAD_TO_DEG
from lunanav.sim.simulator import SimParams, RigidBody, run_sim, SimResults, reverse_sim_results
from lunanav.sim.sensors import (
    SensorEnvironment, Sensor, SensorSuite, SensorName,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor
)
from lunanav.estimation.ekf import ekf_predict, ekf_update, Qd_from_accel_white
from lunanav.sim.quaternion import unitize_state, angle_axis_to_q, quat_apply, conj
from lunanav.plotting import plot_state_vector
from lunanav.visualization import visualize_trajectory, plot_measurements, plot_attitude_relative_vertical, plot_filter_confidence
from lunanav.sim.sensors import get_los_vectors

# Simulation Setup

In [ ]:
@jax.jit
def control_fn(t, state):
    """Simple control: thrust and roll torque"""
    del state
    force_N = jnp.zeros(3)
    torque_Nm = jnp.zeros(3)

    # Thrust profile
    fz = jnp.where(t < 5,   200.0,
         jnp.where(t < 30,  500.0,
         jnp.where(t < 80, 1000.0,
         jnp.where(t < 150, 300.0, 0.0))))
    force_N = jnp.array([0.0, 0.0, fz])

    # Roll torque
    tx = jnp.where(t < 6,   0.005,
         jnp.where(t < 20, -0.003,
         jnp.where(t < 40,  0.002,
         jnp.where(t < 100, -0.001, 0.0))))
    torque_Nm = jnp.array([tx, 0.0, 0.0])

    return force_N, torque_Nm

In [ ]:
# Simulation parameters
lander = RigidBody(
    mass_kg=100,
    I=np.eye(3)
)

dt = 0.1
t_max = 120
nsteps = int(t_max / dt)

state0 = np.array([
    0, 0, R_MOON + 5,      # position [m]
    0, 0, 0,               # velocity [m/s]
    1, 0, 0, 0,            # quaternion
    0, 0, 0                # angular velocity [rad/s]
])

sim = SimParams(state0, lander, dt, t_max)
print(f"Running simulation for {t_max} seconds, {nsteps} steps")

In [ ]:
# Run simulation
liftoff_results = run_sim(state0, nsteps, dt, control_fn, sim)
n = liftoff_results.nsteps
print(f"Simulation completed: {n} steps")

landing_results = reverse_sim_results(liftoff_results)
print("Reversed Results")

results = landing_results
# results = liftoff_results

# New Sensor Architecture: EKF with SensorEnvironment

In [ ]:
# Noise standard deviations (super noisy!)
sigma_accel = 1      # m/s^2
sigma_gyro = 1e-3      # rad/s
sigma_los = 10        # m (super noisy!)
sigma_los_vel = 10     # m/s (super noisy!)
sigma_star = 1e-3      # unitless
sigma_doppler = 0.1      # m/s

# Create sensor suite with new architecture
sensor_suite = SensorSuite(sensors={
    SensorName.ACCELEROMETER: accelerometer_sensor(sigma_accel),
    SensorName.GYROSCOPE: gyroscope_sensor(sigma_gyro),
    SensorName.LASER_ALTIMETER: laser_altimeter_sensor(sigma_los),
    SensorName.LASER_VELOCITY: laser_velocity_sensor(sigma_los_vel),
    SensorName.STAR_TRACKER: star_tracker_sensor(sigma_star),
    SensorName.DOPPLER: doppler_sensor(3, sigma_doppler),
})

for name_enum, sensor in sensor_suite.sensors.items():
    print(f"  {name_enum.value}: {sensor.meas_dim}D, sigma={sensor.noise_cov[0,0]**0.5:.4f}")

In [ ]:
@dataclass
class SatPosVel:
    r: np.ndarray
    v: np.ndarray

def make_sat_arrs(t_arr, altitude, raan, aop, inc) -> SatPosVel:
    """
    Generate satellite trajectory arrays for circular lunar orbit.
    
    Args:
        dt: timestep [s]
        n: number of steps
        altitude for circular orbit [m]
        raan: [deg]
        aop: arg of perigee (also anomaly for circular orbit) from raan [deg]
        inclination: [degrees]
    
    Returns:
        t_arr: time array [n]
        r_arr: position array [n, 3] in inertial frame [m]
        v_arr: velocity array [n, 3] in inertial frame [m/s]
    """
    r_orbit = R_MOON + altitude
    v_norm = np.sqrt(GM_MOON / r_orbit)
    n_mean = np.sqrt(GM_MOON / r_orbit**3) # mean motion
    n = len(t_arr)
    
    inc, raan, aop = np.radians([inc, raan, aop])
    
    r_arr = np.zeros((n, 3))
    v_arr = np.zeros((n, 3))
    
    # inc around x
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(inc), -np.sin(inc)],
        [0, np.sin(inc), np.cos(inc)]])
    
    # raan around z
    Rz = np.array([
        [np.cos(raan), -np.sin(raan), 0],
        [np.sin(raan), np.cos(raan), 0],
        [0, 0, 1]])
    
    # Combined rotation
    R = Rz @ Rx
    
    for i in range(n):
        nu = aop + n_mean * t_arr[i]
        
        # Position and velocity in orbital plane
        r_orb = np.array([r_orbit * np.cos(nu), r_orbit * np.sin(nu), 0])
        v_orb = np.array([-v_norm * np.sin(nu), v_norm * np.cos(nu), 0])
        
        # Apply rotation
        r_arr[i] = R @ r_orb
        v_arr[i] = R @ v_orb
    
    return SatPosVel(r_arr, v_arr)

In [ ]:
doppler_sats: list[SatPosVel] = [
    make_sat_arrs(results.t, altitude=100e3, raan=0, aop=90, inc=90), # overhead going -x
    make_sat_arrs(results.t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86),
]
r_sats = jnp.array([s.r for s in doppler_sats])
v_sats = jnp.array([s.v for s in doppler_sats])

plt.figure(figsize=(8,8))
for sat in doppler_sats:
    plt.plot(sat.r[:,0], sat.r[:,1], alpha=0.7)

plt.xlim(-R_MOON, R_MOON)
plt.ylim(-R_MOON, R_MOON)
plt.tight_layout()
plt.show()

In [ ]:
# Generate synthetic measurements from true trajectory with noise
def generate_measurements(results: SimResults, sensor_suite: SensorSuite, sats: list[SatPosVel] = None):
    """Generate noisy measurements for all timesteps
    
    Returns:
        measurements_clean: Dict[SensorName] -> array of shape (n_steps, meas_dim)
        measurements_noisy: Dict[SensorName] -> array of shape (n_steps, meas_dim)
    """
    n_steps = len(results.t)
    measurements_clean = {}
    measurements_noisy = {}
    
    # Initialize arrays for each sensor
    for sensor_name_enum, sensor in sensor_suite.sensors.items():
        measurements_clean[sensor_name_enum] = np.zeros((n_steps, sensor.meas_dim))
        measurements_noisy[sensor_name_enum] = np.zeros((n_steps, sensor.meas_dim))
    
    for i in tqdm(range(n_steps)):
        state = results.states[i]
        t = results.t[i]
        env = SensorEnvironment(
            t=t,
            mass=sim.body.mass_kg,
            specific_force_body=results.force_N[i],
        )
        if sats is not None:
            env.satellite_positions = r_sats[i]
            env.satellite_velocities = v_sats[i]
        
        for sensor_name_enum, sensor in sensor_suite.sensors.items():
            # Get clean measurement
            z_clean = sensor.measure(state, env)
            measurements_clean[sensor_name_enum][i] = z_clean
            
            # Add noise for sensor measurement
            noise = np.random.multivariate_normal(
                np.zeros(sensor.meas_dim), 
                sensor.get_noise_cov(env)
            )
            z_noisy = np.array(z_clean) + noise
            measurements_noisy[sensor_name_enum][i] = z_noisy
    
    return measurements_clean, measurements_noisy

measurements_clean, measurements_noisy = generate_measurements(results, sensor_suite, doppler_sats)
print(f"Generated {len(measurements_noisy)} measurement sets with noise")

In [ ]:
# Plot all measurements using visualization utility
fig = plot_measurements(measurements_clean, measurements_noisy, results, sensor_suite)
plt.show()

print("\nMeasurement statistics:")
print(f"Laser altitude noise: σ = {sigma_los:.1f} m")
print(f"Laser velocity noise: σ = {sigma_los_vel:.1f} m/s")
laser_alt_clean = measurements_clean[SensorName.LASER_ALTIMETER]
laser_alt_noisy = measurements_noisy[SensorName.LASER_ALTIMETER]
laser_vel_clean = measurements_clean[SensorName.LASER_VELOCITY]
laser_vel_noisy = measurements_noisy[SensorName.LASER_VELOCITY]
print(f"Max laser alt noise: {np.std(laser_alt_noisy - laser_alt_clean):.1f} m")
print(f"Max laser vel noise: {np.std(laser_vel_noisy - laser_vel_clean):.1f} m/s")

# EKF with Sequential Sensor Updates

In [ ]:
# EKF setup
mu_arr = np.zeros((n, 13))
Sigma_arr = np.zeros((n, 13, 13))

state0 = results.states[0]

# Initial state estimate (slightly off from truth)
mu_arr[0] = state0 + np.array([1, 1, 10, 0.1, 0.1, 0.1, *angle_axis_to_q(0, [0,1,0], True), 0.01, 0.01, 0.01])

Sigma_arr[0] = np.eye(13) * 1e-2  # small initial uncertainty

# Process noise
Q_ekf = np.zeros((13, 13))
Q_ekf[0:6, 0:6] = Qd_from_accel_white(dt, sigma_accel)
Q_ekf[6:10, 6:10] = np.eye(4) * 1e-6
Q_ekf[10:13, 10:13] = np.eye(3) * sigma_gyro**2 * dt

print("EKF initialized")
print(f"Initial position error: {norm(mu_arr[0, 0:3] - state0[0:3]):.2f} m")

In [ ]:
# DIAGNOSTICS: Check trajectory and measurement dimensions
print("=== TRAJECTORY DIAGNOSTICS ===")
print(f"liftoff_results.nsteps: {liftoff_results.nsteps}")
print(f"landing_results.nsteps: {landing_results.nsteps}")
print(f"n (current): {n}")
print(f"results.t length: {len(results.t)}")
print(f"results.states shape: {results.states.shape}")
print(f"results.force_N shape: {results.force_N.shape}")

print("\nMeasurement arrays shape:")
for sensor_name, meas_array in measurements_clean.items():
    print(f"  {sensor_name.value}: {meas_array.shape}")

print("\nEKF arrays shape:")
print(f"  mu_arr: {mu_arr.shape}")
print(f"  Sigma_arr: {Sigma_arr.shape}")

print("\nInitial state check:")
print(f"  state0 shape: {state0.shape}")
print(f"  state0: {state0}")
print(f"  mu_arr[0]: {mu_arr[0]}")

In [ ]:
# Run EKF with new sensor architecture using NOISY measurements
# Sequential updates: predict, then update with available sensors

for i in tqdm(range(n - 1)):
    # Predict step
    force_B = results.force_N[i]
    torque_B = results.torque_Nm[i]
    accel_meas = force_B / lander.mass_kg
    gyro_meas = results.states[i, 10:13]
    
    mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
    mu_pred = unitize_state(mu_pred)
    
    # Create environment for this timestep
    env = SensorEnvironment(
        t=results.t[i],
        mass=sim.body.mass_kg,
        specific_force_body=jnp.array(accel_meas),
    )

    # sensor = sensor_suite.sensors[SensorName.ACCELEROMETER]
    # z = measurements_noisy[SensorName.ACCELEROMETER][i]
    # h_pred = sensor.measure(mu_pred, env)
    # H = sensor.jacobian(mu_pred, env)
    # R = sensor.get_noise_cov(env)
    # mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
    # mu_pred = unitize_state(mu_pred)

    # # Gyroscope: every step (very clean measurement)
    # sensor = sensor_suite.sensors[SensorName.GYROSCOPE]
    # z = measurements_noisy[SensorName.GYROSCOPE][i]
    # h_pred = sensor.measure(mu_pred, env)
    # H = sensor.jacobian(mu_pred, env)
    # R = sensor.get_noise_cov(env)
    # mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
    # mu_pred = unitize_state(mu_pred)
    
    # Sequential sensor updates with NOISY measurements
    # Laser altimeter: every 10 steps
    if i % 10 == 0:
        sensor = sensor_suite.sensors[SensorName.LASER_ALTIMETER]
        z = measurements_noisy[SensorName.LASER_ALTIMETER][i]
        h_pred = sensor.measure(mu_pred, env)
        H = sensor.jacobian(mu_pred, env)
        R = sensor.get_noise_cov(env)
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
        mu_pred = unitize_state(mu_pred)
    
    # Laser velocity: every 10 steps
    if i % 10 == 0:
        sensor = sensor_suite.sensors[SensorName.LASER_VELOCITY]
        z = measurements_noisy[SensorName.LASER_VELOCITY][i]
        h_pred = sensor.measure(mu_pred, env)
        H = sensor.jacobian(mu_pred, env)
        R = sensor.get_noise_cov(env)
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
        mu_pred = unitize_state(mu_pred)
    
    # Star tracker: every 50 steps
    if i % 50 == 0:
        sensor = sensor_suite.sensors[SensorName.STAR_TRACKER]
        z = measurements_noisy[SensorName.STAR_TRACKER][i]
        h_pred = sensor.measure(mu_pred, env)
        H = sensor.jacobian(mu_pred, env)
        R = sensor.get_noise_cov(env)
        mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
        mu_pred = unitize_state(mu_pred)

        # Star tracker: every 50 steps
    sensor = sensor_suite.sensors[SensorName.DOPPLER]
    z = measurements_noisy[SensorName.DOPPLER][i]
    h_pred = sensor.measure(mu_pred, env)
    H = sensor.jacobian(mu_pred, env)
    R = sensor.get_noise_cov(env)
    mu_pred, Sigma_pred = ekf_update(mu_pred, Sigma_pred, z, H, h_pred, R)
    mu_pred = unitize_state(mu_pred)
    
    
    mu_arr[i + 1] = mu_pred
    Sigma_arr[i + 1] = Sigma_pred
    
    # if i % 100 == 0:
    #     print(f"Step {i}/{n}")

print("EKF completed with SUPER NOISY measurements")

# Results & Analysis

In [ ]:
# Position errors
pos_errors = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
vel_errors = np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(results.t, pos_errors, label='Position Error')
ax1.set_ylabel('Position Error (m)')
ax1.set_title('EKF Estimation Error')
ax1.grid()
ax1.legend()

ax2.plot(results.t, vel_errors, label='Velocity Error')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Velocity Error (m/s)')
ax2.grid()
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final position error: {pos_errors[-1]:.2f} m")
print(f"Final velocity error: {vel_errors[-1]:.4f} m/s")
print(f"Max position error: {np.max(pos_errors):.2f} m")

In [ ]:

# Add this to your notebook as a new cell:
fig = plot_filter_confidence(Sigma_arr, results.t)
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("FILTER CONFIDENCE SUMMARY")
print("="*60)

trace_initial = np.trace(Sigma_arr[0])
trace_final = np.trace(Sigma_arr[-1])
trace_min = np.min([np.trace(Sigma_arr[i]) for i in range(len(Sigma_arr))])
trace_max = np.max([np.trace(Sigma_arr[i]) for i in range(len(Sigma_arr))])

print(f"\nOverall Uncertainty (Trace of Σ):")
print(f"  Initial: {trace_initial:.4e}")
print(f"  Final:   {trace_final:.4e}")
print(f"  Min:     {trace_min:.4e}")
print(f"  Max:     {trace_max:.4e}")

# Component-wise final uncertainties
print(f"\nFinal Standard Deviations by Component:")
print(f"  Position:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 0:3, 0:3]))):.2f} m")
print(f"  Velocity:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 3:6, 3:6]))):.4f} m/s")
print(f"  Attitude:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 6:10, 6:10]))):.6f}")
print(f"  Angular Velocity:  {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 10:13, 10:13]))):.6f} rad/s")

# Convergence metric
if trace_final < trace_initial:
    improvement = (trace_initial - trace_final) / trace_initial * 100
    print(f"\n✓ Filter CONVERGED: {improvement:.1f}% reduction in uncertainty")
else:
    increase = (trace_final - trace_initial) / trace_initial * 100
    print(f"\n✗ Filter DIVERGED: {increase:.1f}% increase in uncertainty")

In [ ]:
# Detailed component-wise error analysis to diagnose divergence
pos_error_x = mu_arr[:, 0] - results.states[:, 0]
pos_error_y = mu_arr[:, 1] - results.states[:, 1]
pos_error_z = mu_arr[:, 2] - results.states[:, 2]

vel_error_x = mu_arr[:, 3] - results.states[:, 3]
vel_error_y = mu_arr[:, 4] - results.states[:, 4]
vel_error_z = mu_arr[:, 5] - results.states[:, 5]

fig, axes = plt.subplots(3, 2, figsize=(15, 11))

# Position components
axes[0, 0].plot(results.t, pos_error_x, 'r-', label='X error', linewidth=1)
axes[0, 0].plot(results.t, pos_error_y, 'g-', label='Y error', linewidth=1)
axes[0, 0].plot(results.t, pos_error_z, 'b-', label='Z error', linewidth=1)
axes[0, 0].set_ylabel('Position Error (m)')
axes[0, 0].set_title('Position Error - Component-wise')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

axes[0, 1].semilogy(results.t, np.abs(pos_error_x) + 1, 'r-', label='|X error|', linewidth=1)
axes[0, 1].semilogy(results.t, np.abs(pos_error_y) + 1, 'g-', label='|Y error|', linewidth=1)
axes[0, 1].semilogy(results.t, np.abs(pos_error_z) + 1, 'b-', label='|Z error|', linewidth=1)
axes[0, 1].set_ylabel('Position Error (m, log scale)')
axes[0, 1].set_title('Position Error - Log Scale')
axes[0, 1].grid(alpha=0.3, which='both')
axes[0, 1].legend()

# Velocity components
axes[1, 0].plot(results.t, vel_error_x, 'r-', label='VX error', linewidth=1)
axes[1, 0].plot(results.t, vel_error_y, 'g-', label='VY error', linewidth=1)
axes[1, 0].plot(results.t, vel_error_z, 'b-', label='VZ error', linewidth=1)
axes[1, 0].set_ylabel('Velocity Error (m/s)')
axes[1, 0].set_title('Velocity Error - Component-wise')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(results.t, np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1), 'k-', linewidth=2)
axes[1, 1].set_ylabel('Velocity Error Norm (m/s)')
axes[1, 1].set_title('Total Velocity Error')
axes[1, 1].grid(alpha=0.3)

# Altitude comparison and error
altitude_true = results.states[:, 2]
altitude_est = mu_arr[:, 2]
axes[2, 0].plot(results.t, altitude_true, 'k-', label='True altitude', linewidth=2)
axes[2, 0].plot(results.t, altitude_est, 'r--', label='Estimated altitude', linewidth=2)
axes[2, 0].set_ylabel('Altitude (m)')
axes[2, 0].set_title('Altitude: Truth vs Estimate')
axes[2, 0].grid(alpha=0.3)
axes[2, 0].legend()

axes[2, 1].plot(results.t, altitude_est - altitude_true, 'r-', linewidth=2, label='Altitude error')
axes[2, 1].set_ylabel('Altitude Error (m)')
axes[2, 1].set_xlabel('Time (s)')
axes[2, 1].set_title('Altitude Error Over Time')
axes[2, 1].grid(alpha=0.3)
axes[2, 1].legend()

plt.tight_layout()
plt.show()

# Detailed statistics
print("\n" + "="*50)
print("DIVERGENCE ANALYSIS")
print("="*50)
print(f"\nPosition Error Statistics:")
print(f"  Initial: {np.linalg.norm(mu_arr[0, 0:3] - results.states[0, 0:3]):.2f} m")
print(f"  Final:   {np.linalg.norm(mu_arr[-1, 0:3] - results.states[-1, 0:3]):.2f} m")
print(f"  Max:     {np.max(np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)):.2f} m")
print(f"  Mean:    {np.mean(np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)):.2f} m")

print(f"\nVelocity Error Statistics:")
print(f"  Initial: {np.linalg.norm(mu_arr[0, 3:6] - results.states[0, 3:6]):.4f} m/s")
print(f"  Final:   {np.linalg.norm(mu_arr[-1, 3:6] - results.states[-1, 3:6]):.4f} m/s")
print(f"  Max:     {np.max(np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)):.4f} m/s")
print(f"  Mean:    {np.mean(np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)):.4f} m/s")

# Find divergence onset
pos_norms = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
for threshold in [100, 500, 1000, 5000]:
    idx = np.argmax(pos_norms > threshold)
    if idx > 0:
        print(f"\nError exceeds {threshold}m at step {idx}, t={results.t[idx]:.1f}s, error={pos_norms[idx]:.1f}m")
        break

print(f"\nTrajectory length: {n} steps ({results.t[-1]:.1f}s)")
print(f"Number of measurement updates: {len([i for i in range(n) if i % 10 == 0])} (every 10 steps)")

In [ ]:
plot_state_vector(
    results.t, 
    mu_arr[:, 0:3], 
    mu_arr[:, 3:6], 
    mu_arr[:, 10:13],
    figsize=(14, 10)
)

# State estimates vs truth
plot_state_vector(
    results.t, 
    mu_arr[:, 0:3] - results.states[:, 0:3], 
    mu_arr[:, 3:6] - results.states[:, 3:6], 
    mu_arr[:, 10:13] - results.states[:, 10:13],
    figsize=(14, 10)
)

In [ ]:
# Quaternion error
q_error = np.linalg.norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)

plt.figure(figsize=(10, 5))
plt.plot(results.t, q_error)
plt.xlabel('Time (s)')
plt.ylabel('Quaternion Error')
plt.title('Attitude Estimation Error')
plt.grid()
plt.show()

In [ ]:
# Plot attitude relative to vertical for both truth and estimate
fig_true, tilt_true = plot_attitude_relative_vertical(results.states, results.t)
plt.suptitle("True Trajectory - Attitude Relative to Vertical", fontsize=14, y=1.00)
plt.show()

fig_est, tilt_est = plot_attitude_relative_vertical(mu_arr, results.t)
plt.suptitle("EKF Estimate - Attitude Relative to Vertical", fontsize=14, y=1.00)
plt.show()

# Plot both together
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(results.t, tilt_true, 'k-', linewidth=2, label='True')
ax.plot(results.t, tilt_est, 'r--', linewidth=2, label='Estimated')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Tilt Angle (degrees)', fontsize=12)
ax.set_title('Lander Tilt Angle: Truth vs EKF Estimate', fontsize=14)
ax.grid(alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
moon_offset =  np.tile([0,0,R_MOON,0,0,0,0,0,0,0,0,0,0], (n, 1))
visualize_trajectory([results.states - moon_offset, mu_arr - moon_offset], results.t, dt, title="EKF Estimated Trajectory with LOS Vectors", show_lander=False, downsample_rate=20).show()

In [ ]:
from numpy.linalg import svd

def obsv_verbose(x, sensors, a_m, w_m, Q, sim, env, h=3, show_plot = False):
    def num_jac(f, x):
        x = np.array(x, dtype=float)
        f0 = f(x)
        if np.isscalar(f0): f0 = np.array([f0])
        J = np.zeros((len(f0), len(x)))
        for i in range(len(x)):
            e = np.zeros(13); e[i] = 1e-5
            J[:, i] = (f(np.array(x + e, dtype=float)) - f(np.array(x - e, dtype=float))) / 2e-5
        return J
    O, F_n = [], np.eye(13)
    for _ in range(h):
        H = np.vstack([num_jac(lambda s, sensor=sensor: sensor.measure(s, env), x) for sensor in sensors.sensors.values()])
        O.append(H @ F_n)
        F = num_jac(lambda s: ekf_predict(np.array(s, dtype=float), np.eye(13), a_m, w_m, Q, sim)[0], x)
        x = unitize_state(ekf_predict(np.array(x, dtype=float), np.eye(13), a_m, w_m, Q, sim)[0])
        F_n = F_n @ F
    U, S, V = svd(np.vstack(O))
    r = np.sum(S > S[0]*1e-6)
    
    print(f"Rank {r}/13\n")
    print("OBSERVABLE states:")
    state_names = ["x", "y", "z", "vx", "vy", "vz", "q0", "q1", "q2", "q3", "ωx", "ωy", "ωz"]
    for i in range(r):
        print(f"  Mode {i}: S={S[i]:.2e}")
    
    print(f"\nUNOBSERVABLE states ({13-r}):")
    for i in range(r, len(S)):
        null_vec = V[i, :]
        contribs = np.abs(null_vec)
        top_idx = np.argsort(contribs)[-1]
        print(f"  Mode {i}: Primary = {state_names[top_idx]} ({null_vec[top_idx]:.3f})")

    if show_plot:
        
        plt.figure(figsize=(10, 4))
        plt.semilogy(S, 'ko-', linewidth=2, markersize=6)
        plt.axhline(S[0]*1e-6, linewidth=2, label='Rank threshold')
        plt.xlabel('Singular Value Index'); plt.ylabel('Singular Value')
        plt.title(f'Observability (Rank {r}/13)'); plt.grid(); plt.legend(); plt.show()

for t_idx in [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100]:
    obsv_verbose(mu_arr[t_idx], sensor_suite, results.force_N[t_idx]/lander.mass_kg, results.states[t_idx,10:13], Q_ekf, sim, SensorEnvironment(t=results.t[t_idx], mass=sim.body.mass_kg))
